# Classical Entailment Reasoning Agent

This notebook builds a symbolic forward-chaining reasoner. It does not use a language model to decide labels. The included English parser covers a deliberately bounded subset of controlled statements; unsupported constructions are reported rather than silently treated as successfully parsed. FOLIO contains broader first-order logic and requires richer parsing for full coverage.

The inference target is positive Horn entailment. A query is predicted `True` when it is derivable; otherwise it is `False` under the benchmarks' binary-label convention. This is not a complete open-world first-order theorem prover: absence of a derivation is not proof of logical negation.

In [12]:
# 1. Import Required Libraries
from __future__ import annotations

import json
import re
import time
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable, Optional

## 2. Load and Explore Datasets

The Hugging Face `datasets` package is optional. Set `LOAD_PUBLIC_DATASETS = True` to download public splits. Repository IDs and split names are configurable because dataset mirrors/configurations can change; inspect the printed schema before evaluating. No private test data is read here.

In [13]:
# Set to True in an environment with network access and `datasets` installed.
LOAD_PUBLIC_DATASETS = False
PUBLIC_DATASET_REPOS = {
    "RuleTaker": "allenai/RuleTaker",
    "CounterBench": "allenai/CounterBench",
    "FOLIO": "yale-nlp/FOLIO",
}
PUBLIC_SPLITS = {"RuleTaker": "validation", "CounterBench": "validation", "FOLIO": "validation"}
public_datasets: dict[str, Any] = {}


def load_public_datasets() -> dict[str, Any]:
    try:
        from datasets import load_dataset
    except ImportError as exc:
        raise RuntimeError("Install `datasets` to load Hugging Face datasets.") from exc

    loaded = {}
    for name, repo_id in PUBLIC_DATASET_REPOS.items():
        split = PUBLIC_SPLITS[name]
        try:
            dataset = load_dataset(repo_id, split=split)
            loaded[name] = dataset
            print(f"{name}: {len(dataset)} rows; columns={dataset.column_names}")
            if len(dataset):
                print("  example keys:", list(dataset[0].keys()))
        except Exception as exc:
            print(f"Could not load {name} ({repo_id}, split={split}): {exc}")
            print("  Check the Hugging Face repository/config and split, then update the constants above.")
    return loaded


if LOAD_PUBLIC_DATASETS:
    public_datasets = load_public_datasets()
else:
    print("Dataset downloads disabled. Set LOAD_PUBLIC_DATASETS=True to inspect the public splits.")

Dataset downloads disabled. Set LOAD_PUBLIC_DATASETS=True to inspect the public splits.


## 3. Parse Knowledge Base: Facts and Rules

The parser handles simple property facts, binary relations, explicit negated facts, conjunctions, and universally quantified `If ... then ...` rules expressed in controlled English. Unsupported wording raises `ParseError`, allowing evaluation to count coverage instead of guessing.

In [14]:
class ParseError(ValueError):
    pass


UNIVERSAL_TERMS = {"someone", "somebody", "something", "anyone", "anybody", "everyone", "everybody", "they", "he", "she", "it"}


def _term(text: str) -> str:
    cleaned = re.sub(r"^(a|an|the)\s+", "", text.strip(), flags=re.IGNORECASE)
    normalized = re.sub(r"\s+", "_", cleaned.lower())
    return "?x" if cleaned.lower() in UNIVERSAL_TERMS else normalized


def _sentences(text: Any) -> list[str]:
    if isinstance(text, (list, tuple)):
        text = " ".join(str(part) for part in text)
    if not isinstance(text, str):
        raise ParseError(f"Expected text, got {type(text).__name__}")
    return [part.strip() for part in re.split(r"[.?!\n]+", text) if part.strip()]


def parse_atom_clause(text: str) -> "Atom":
    clause = re.sub(r"[,;]+", " ", text.strip().rstrip(".?!"))
    clause = re.sub(r"\s+", " ", clause)
    negated = False
    if clause.lower().startswith("not "):
        negated = True
        clause = clause[4:].strip()

    copula = re.match(r"^(.+?)\s+(is|are|was|were)\s+(.+)$", clause, flags=re.IGNORECASE)
    if copula:
        subject, _, complement = copula.groups()
        complement = complement.strip()
        if complement.lower().startswith("not "):
            negated = not negated
            complement = complement[4:].strip()
        complement = re.sub(r"^(a|an|the)\s+", "", complement, flags=re.IGNORECASE)
        predicate = re.sub(r"\s+", "_", complement.lower())
        if not predicate:
            raise ParseError(f"Missing predicate in: {text}")
        return Atom(predicate, (_term(subject),), negated)

    relation = re.match(r"^(\S+)\s+([a-z][a-z-]*)\s+(.+)$", clause, flags=re.IGNORECASE)
    if relation:
        subject, predicate, object_term = relation.groups()
        return Atom(predicate.lower(), (_term(subject), _term(object_term)), negated)
    raise ParseError(f"Unsupported clause: {text}")


def _expand_conjunction(text: str) -> list[str]:
    pieces = [piece.strip() for piece in re.split(r"\s+and\s+", text, flags=re.IGNORECASE)]
    expanded = []
    frame = None
    for piece in pieces:
        if not re.search(r"\b(is|are|was|were)\b", piece, flags=re.IGNORECASE) and frame:
            piece = frame + piece
        expanded.append(piece)
        match = re.match(r"^(.+?\s+(?:is|are|was|were)\s+)", piece, flags=re.IGNORECASE)
        if match:
            frame = match.group(1)
    return expanded


def parse_knowledge_base(context: Any) -> tuple[list["Atom"], list["HornRule"], list[str]]:
    facts, rules, unsupported = [], [], []
    for sentence in _sentences(context):
        rule_match = re.match(r"^if\s+(.+?)\s*,?\s+then\s+(.+)$", sentence, flags=re.IGNORECASE)
        try:
            if rule_match:
                body_text, head_text = rule_match.groups()
                body = tuple(parse_atom_clause(part) for part in _expand_conjunction(body_text))
                head_parts = _expand_conjunction(head_text)
                if len(head_parts) != 1:
                    raise ParseError("Conjunctive rule consequents are not supported")
                rules.append(HornRule(body, parse_atom_clause(head_parts[0])))
            else:
                facts.extend(parse_atom_clause(part) for part in _expand_conjunction(sentence))
        except ParseError as exc:
            unsupported.append(f"{sentence}: {exc}")
    return facts, rules, unsupported


def parse_query(query: Any) -> "Atom":
    sentences = _sentences(query)
    if len(sentences) != 1:
        raise ParseError("A query must contain exactly one statement")
    return parse_atom_clause(sentences[0])

## 4. Internal Representation of Facts and Rules

Atoms store a predicate, ordered terms, and an explicit sign. Terms beginning with `?` are variables. A Horn rule contains a conjunctive body and one head atom; rule variables are grounded by unification during inference.

In [15]:
@dataclass(frozen=True)
class Atom:
    predicate: str
    arguments: tuple[str, ...]
    negated: bool = False

    def __str__(self) -> str:
        sign = "not " if self.negated else ""
        args = ", ".join(self.arguments)
        return f"{sign}{self.predicate}({args})"


@dataclass(frozen=True)
class HornRule:
    body: tuple[Atom, ...]
    head: Atom

    def __str__(self) -> str:
        return f"{', '.join(map(str, self.body))} -> {self.head}"


@dataclass
class KnowledgeBase:
    facts: set[Atom]
    rules: list[HornRule]


def make_knowledge_base(facts: Iterable[Atom], rules: Iterable[HornRule]) -> KnowledgeBase:
    return KnowledgeBase(set(facts), list(rules))

## 5. Reasoning Engine Implementation

The engine repeatedly joins each rule body against currently known signed ground atoms, using first-order unification for variables. Newly instantiated heads are added until a fixed point is reached. The finite Herbrand domain is the constants already present in the facts; no generative or neural inference is involved.

In [16]:
def unify(pattern: Atom, ground: Atom, bindings: Optional[dict[str, str]] = None) -> Optional[dict[str, str]]:
    result = dict(bindings or {})
    if pattern.predicate != ground.predicate or pattern.negated != ground.negated:
        return None
    if len(pattern.arguments) != len(ground.arguments):
        return None
    for term, value in zip(pattern.arguments, ground.arguments):
        if term.startswith("?"):
            if term in result and result[term] != value:
                return None
            result[term] = value
        elif term != value:
            return None
    return result


def _instantiate(atom: Atom, bindings: dict[str, str]) -> Optional[Atom]:
    arguments = []
    for term in atom.arguments:
        if term.startswith("?"):
            if term not in bindings:
                return None
            arguments.append(bindings[term])
        else:
            arguments.append(term)
    return Atom(atom.predicate, tuple(arguments), atom.negated)


def _body_bindings(body: tuple[Atom, ...], facts: set[Atom]):
    def visit(index: int, bindings: dict[str, str]):
        if index == len(body):
            yield bindings
            return
        for fact in facts:
            updated = unify(body[index], fact, bindings)
            if updated is not None:
                yield from visit(index + 1, updated)
    yield from visit(0, {})


def forward_chain(kb: KnowledgeBase, max_rounds: int = 1000) -> tuple[set[Atom], list[tuple[HornRule, Atom]]]:
    known = set(kb.facts)
    if any(term.startswith("?") for atom in known for term in atom.arguments):
        raise ValueError("Knowledge-base facts must be ground")
    trace: list[tuple[HornRule, Atom]] = []

    for _ in range(max_rounds):
        additions = set()
        for rule in kb.rules:
            for bindings in _body_bindings(rule.body, known):
                conclusion = _instantiate(rule.head, bindings)
                if conclusion is not None and conclusion not in known:
                    additions.add(conclusion)
        if not additions:
            return known, trace
        for conclusion in sorted(additions, key=str):
            supporting_rule = next(
                rule for rule in kb.rules
                if any(_instantiate(rule.head, bindings) == conclusion
                       for bindings in _body_bindings(rule.body, known))
            )
            trace.append((supporting_rule, conclusion))
        known.update(additions)
    raise RuntimeError(f"Forward chaining exceeded {max_rounds} rounds")

## 6. Query Processing and Entailment Checking

Text entry points return a prediction only when the context and query are within the parser's supported fragment. This makes coverage measurable and prevents malformed or unsupported examples from being counted as negative entailments.

In [17]:
def predict_from_text(context: Any, query_text: Any) -> dict[str, Any]:
    facts, rules, unsupported = parse_knowledge_base(context)
    if unsupported:
        return {"prediction": None, "status": "unsupported_context", "details": unsupported, "trace": []}
    try:
        query_atom = parse_query(query_text)
    except ParseError as exc:
        return {"prediction": None, "status": "unsupported_query", "details": [str(exc)], "trace": []}
    if any(term.startswith("?") for term in query_atom.arguments):
        return {"prediction": None, "status": "variable_query", "details": ["Queries must name a ground entity"], "trace": []}

    closure, trace = forward_chain(make_knowledge_base(facts, rules))
    return {
        "prediction": query_atom in closure,
        "status": "ok",
        "details": [],
        "trace": [f"{rule} ==> {conclusion}" for rule, conclusion in trace],
    }


FIELD_ALIASES = {
    "id": ("id", "ID", "example_id", "uid"),
    "context": ("context", "facts", "premises", "story"),
    "query": ("question", "query", "statement", "hypothesis", "conclusion"),
    "label": ("label", "answer", "gold_label", "entailment", "target"),
}


def _get_field(row: dict[str, Any], aliases: tuple[str, ...], default=None):
    lowered = {str(key).lower(): value for key, value in row.items()}
    for alias in aliases:
        if alias.lower() in lowered:
            return lowered[alias.lower()]
    return default


def normalize_label(value: Any) -> Optional[bool]:
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, float)) and value in (0, 1):
        return bool(value)
    normalized = str(value).strip().lower()
    if normalized in {"true", "yes", "entailed", "entailment", "1"}:
        return True
    if normalized in {"false", "no", "not entailed", "refuted", "0"}:
        return False
    return None


def predict_record(row: dict[str, Any]) -> dict[str, Any]:
    example_id = _get_field(row, FIELD_ALIASES["id"])
    context = _get_field(row, FIELD_ALIASES["context"])
    query = _get_field(row, FIELD_ALIASES["query"])
    if context is None or query is None:
        return {"id": example_id, "gold": normalize_label(_get_field(row, FIELD_ALIASES["label"])),
                "prediction": None, "status": "unrecognized_schema", "details": [list(row.keys())], "trace": []}
    result = predict_from_text(context, query)
    result.update({"id": example_id, "gold": normalize_label(_get_field(row, FIELD_ALIASES["label"]))})
    return result

## 7. Batch Inference on Public Dataset

Each row is passed through the same parser and reasoner. The batch records predictions, labels, status, and derivation traces. Unsupported rows remain explicit and are excluded from accuracy while still contributing to coverage counts.

In [18]:
def run_batch(rows: Iterable[dict[str, Any]], limit: Optional[int] = None) -> tuple[list[dict[str, Any]], float]:
    start = time.perf_counter()
    results = []
    for index, row in enumerate(rows):
        if limit is not None and index >= limit:
            break
        results.append(predict_record(dict(row)))
    return results, time.perf_counter() - start


demo_rows = [
    {
        "id": "demo-chain-true",
        "context": "Bob is young. Bob is kind. If someone is young, then they are smart. If someone is smart and kind, then they are successful.",
        "question": "Bob is successful.",
        "label": True,
    },
    {
        "id": "demo-not-derived",
        "context": "Bob is young. If someone is young, then they are smart.",
        "question": "Bob is successful.",
        "label": False,
    },
]

demo_results, demo_elapsed = run_batch(demo_rows)
assert all(result["status"] == "ok" for result in demo_results), demo_results
assert [result["prediction"] for result in demo_results] == [True, False], demo_results
print(f"Synthetic smoke test: {len(demo_results)} examples, {demo_elapsed:.6f}s")
print("Derived trace:", demo_results[0]["trace"])

public_batch_results = {}
for dataset_name, dataset in public_datasets.items():
    results, elapsed = run_batch(dataset)
    public_batch_results[dataset_name] = {"results": results, "elapsed_seconds": elapsed}
    supported = [result for result in results if result["status"] == "ok"]
    print(f"{dataset_name}: {len(supported)}/{len(results)} supported; {elapsed:.3f}s")

Synthetic smoke test: 2 examples, 0.000447s
Derived trace: ['young(?x) -> smart(?x) ==> smart(bob)', 'smart(?x), kind(?x) -> successful(?x) ==> successful(bob)']


## 8. Evaluate Accuracy and Inference Time

Accuracy is calculated only for rows that both parse successfully and have a recognizable gold label. Coverage reports what fraction of rows the parser/reasoner handled. Timing includes parsing and inference for the full batch; public-set metrics are printed only when public datasets were actually loaded.

In [19]:
def summarize_results(results: list[dict[str, Any]], elapsed_seconds: float) -> dict[str, Any]:
    evaluated = [item for item in results if item["status"] == "ok" and item["gold"] is not None]
    correct = sum(item["prediction"] == item["gold"] for item in evaluated)
    summary = {
        "rows": len(results),
        "supported": sum(item["status"] == "ok" for item in results),
        "labeled_supported": len(evaluated),
        "correct": correct,
        "accuracy": correct / len(evaluated) if evaluated else None,
        "coverage": sum(item["status"] == "ok" for item in results) / len(results) if results else 0.0,
        "elapsed_seconds": elapsed_seconds,
        "seconds_per_row": elapsed_seconds / len(results) if results else 0.0,
    }
    return summary


if public_batch_results:
    public_metrics = {}
    for dataset_name, batch in public_batch_results.items():
        metrics = summarize_results(batch["results"], batch["elapsed_seconds"])
        public_metrics[dataset_name] = metrics
        accuracy = "n/a" if metrics["accuracy"] is None else f"{metrics['accuracy']:.2%}"
        print(
            f"{dataset_name}: accuracy={accuracy}, coverage={metrics['coverage']:.2%}, "
            f"supported={metrics['supported']}/{metrics['rows']}, "
            f"inference_time={metrics['elapsed_seconds']:.3f}s"
        )
else:
    demo_metrics = summarize_results(demo_results, demo_elapsed)
    print("No public dataset was loaded; these are synthetic smoke-test metrics, not benchmark accuracy.")
    print(f"Synthetic accuracy={demo_metrics['accuracy']:.2%}; inference_time={demo_metrics['elapsed_seconds']:.6f}s")

No public dataset was loaded; these are synthetic smoke-test metrics, not benchmark accuracy.
Synthetic accuracy=100.00%; inference_time=0.000447s


## 9. Generate Submission File

Provide a CSV of evaluation examples with ID, context, query, and optionally a label column. The notebook does not locate or open a private test file automatically. The writer refuses to emit a partial submission if any row is unsupported, preserving one prediction per supplied ID.

In [20]:
import csv
import tempfile


def generate_submission(rows: Iterable[dict[str, Any]], output_path: str | Path) -> Path:
    rows = list(rows)
    results = [predict_record(dict(row)) for row in rows]
    unsupported = [result for result in results if result["status"] != "ok" or result["prediction"] is None]
    if unsupported:
        statuses = defaultdict(int)
        for item in unsupported:
            statuses[item["status"]] += 1
        raise ValueError(f"Cannot generate complete submission; unsupported rows by status: {dict(statuses)}")

    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    with output.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=["ID", "Label"])
        writer.writeheader()
        for original, result in zip(rows, results):
            example_id = _get_field(dict(original), FIELD_ALIASES["id"])
            if example_id is None:
                raise ValueError("Every submission row must include an ID")
            writer.writerow({"ID": example_id, "Label": str(result["prediction"])})
    return output


def generate_submission_from_csv(input_path: str | Path, output_path: str | Path = "submission/submission.csv") -> Path:
    with Path(input_path).open(newline="", encoding="utf-8") as file:
        rows = list(csv.DictReader(file))
    return generate_submission(rows, output_path)


# Safe writer smoke test in a temporary directory; it does not create a submission file.
with tempfile.TemporaryDirectory() as temp_dir:
    smoke_path = generate_submission(demo_rows, Path(temp_dir) / "submission.csv")
    with smoke_path.open(newline="", encoding="utf-8") as file:
        smoke_rows = list(csv.DictReader(file))
    assert [row["Label"] for row in smoke_rows] == ["True", "False"]
    print(f"Submission writer smoke test passed ({len(smoke_rows)} rows).")

# For an authorized evaluation CSV, run explicitly, for example:
# generate_submission_from_csv("submission/evaluation_input.csv", "submission/submission.csv")

Submission writer smoke test passed (2 rows).
